# 04 — Python & SQL Engineering

**Dataset:** `data/loan_data_04.csv`

This notebook builds a reusable CLI-style ETL flow using modular Python, parameterized SQL, transactions, configuration separation, structured exceptions, staging, merge/upsert, metadata columns, and row-count audit data.

The demonstration uses the repository's PostgreSQL configuration so the hands-on flow matches runtime behavior. It uses SQLAlchemy transactions and parameterized statements; production-scale loading would normally use PostgreSQL `COPY` or an equivalent driver API.

In [17]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if ROOT.name.lower() == "notebooks":
    ROOT = ROOT.parent

DATA_FILE = ROOT / "data" / "loan_data_04.csv"
assert DATA_FILE.exists(), f"Dataset not found: {DATA_FILE}"

raw = pd.read_csv(DATA_FILE)
print(f"Dataset: {DATA_FILE.name}")
print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]}")
print(raw.head(3).to_string(index=False))

Dataset: loan_data_04.csv
Rows: 48 | Columns: 13
 Loan_ID Gender Married Dependents Education Self_Employed  ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  Credit_History Property_Area Loan_Status
LP001778   Male     Yes          1  Graduate            No             3155             1779.0       140.0             360.0             1.0     Semiurban           Y
LP001788 Female      No          0  Graduate           Yes             3463                0.0       122.0             360.0             NaN         Urban           Y
LP001790 Female      No          1  Graduate            No             3812                0.0       112.0             360.0             1.0         Rural           Y


## Learning Content

- Use bind parameters; never construct SQL from untrusted values.
- Keep a related write set inside one transaction.
- Separate extract, transform, load, and validation functions.
- Load environment-specific configuration outside source code.
- Raise structured exceptions and return meaningful CLI exit codes.
- Prefer bulk loading into staging, then merge into the target.
- Attach `run_id`, source file, and timestamps for auditability.

In [18]:
import os
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sqlalchemy import text
from medalloan.config import Settings
from medalloan.database import create_db_engine
from uuid6 import uuid7
from dataclasses import dataclass
from datetime import datetime, timezone

class PipelineError(RuntimeError):
    """Expected pipeline failure."""

class ContractError(PipelineError):
    """Source contract failure."""

@dataclass(frozen=True)
class AppConfig:
    environment: str
    batch_size: int

def load_config() -> AppConfig:
    # Non-secret defaults are acceptable. Secrets must come from a secret manager
    # or environment and are deliberately not printed.
    return AppConfig(
        environment=os.getenv("APP_ENV", "training"),
        batch_size=int(os.getenv("BATCH_SIZE", "1000")),
    )

REQUIRED_COLUMNS = {"Loan_ID", "ApplicantIncome", "CoapplicantIncome",
                    "LoanAmount", "Loan_Status"}

In [19]:
def extract(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    missing = REQUIRED_COLUMNS.difference(frame.columns)
    if missing:
        raise ContractError(f"Missing required columns: {sorted(missing)}")
    return frame

def transform(frame: pd.DataFrame, run_id: str) -> pd.DataFrame:
    result = frame.copy()
    result["Loan_ID"] = result["Loan_ID"].astype("string").str.strip()
    for column in ["ApplicantIncome", "CoapplicantIncome", "LoanAmount"]:
        result[column] = pd.to_numeric(result[column], errors="coerce")
    result["TotalIncome"] = result["ApplicantIncome"] + result["CoapplicantIncome"]
    result["run_id"] = run_id
    result["source_file"] = DATA_FILE.name
    result["loaded_at"] = datetime.now(timezone.utc).isoformat()
    return result.drop_duplicates("Loan_ID", keep="last")

def selected_records(frame: pd.DataFrame):
    columns = ["Loan_ID", "ApplicantIncome", "CoapplicantIncome", "LoanAmount",
               "Loan_Status", "TotalIncome", "run_id", "source_file", "loaded_at"]
    return list(frame[columns].itertuples(index=False, name=None))

## Hands-on / Demonstration

### Build a reusable pipeline with staging and transactional merge

In [20]:
config = load_config()
run_id = str(uuid7())
transformed = transform(extract(DATA_FILE), run_id)

engine = create_db_engine(Settings.from_env())

try:
    with engine.begin() as connection:
        connection.execute(text("""
            CREATE TABLE IF NOT EXISTS loan_target (
                loan_id TEXT PRIMARY KEY,
                applicantincome REAL,
                coapplicantincome REAL,
                loanamount REAL,
                loan_status TEXT,
                totalincome REAL,
                run_id TEXT,
                source_file TEXT,
                loaded_at TEXT
            )
        """))

        connection.execute(text("DROP TABLE IF EXISTS loan_stage"))

        connection.execute(text("""
            CREATE TABLE loan_stage (
                loan_id TEXT,
                applicantincome REAL,
                coapplicantincome REAL,
                loanamount REAL,
                loan_status TEXT,
                totalincome REAL,
                run_id TEXT,
                source_file TEXT,
                loaded_at TEXT
            )
        """))

        columns = [
            "loan_id",
            "applicant_income",
            "coapplicant_income",
            "loan_amount",
            "loan_status",
            "total_income",
            "run_id",
            "source_file",
            "loaded_at",
        ]

        records = [
            dict(zip(columns, row))
            for row in selected_records(transformed)
        ]

        connection.execute(
            text("""
                INSERT INTO loan_stage (
                    loan_id,
                    applicantincome,
                    coapplicantincome,
                    loanamount,
                    loan_status,
                    totalincome,
                    run_id,
                    source_file,
                    loaded_at
                )
                VALUES (
                    :loan_id,
                    :applicant_income,
                    :coapplicant_income,
                    :loan_amount,
                    :loan_status,
                    :total_income,
                    :run_id,
                    :source_file,
                    :loaded_at
                )
            """),
            records,
        )

        connection.execute(text("""
            INSERT INTO loan_target
            SELECT *
            FROM loan_stage
            ON CONFLICT (loan_id) DO UPDATE SET
                applicantincome = EXCLUDED.applicantincome,
                coapplicantincome = EXCLUDED.coapplicantincome,
                loanamount = EXCLUDED.loanamount,
                loan_status = EXCLUDED.loan_status,
                totalincome = EXCLUDED.totalincome,
                run_id = EXCLUDED.run_id,
                source_file = EXCLUDED.source_file,
                loaded_at = EXCLUDED.loaded_at
        """))

except Exception as error:
    raise PipelineError("Transactional load failed") from error

print("Environment:", config.environment)
print("Run ID:", run_id)

with engine.connect() as connection:
    loaded_rows = connection.execute(
        text("SELECT COUNT(*) FROM loan_target")
    ).scalar_one()

print("Loaded rows:", loaded_rows)

Environment: training
Run ID: 01a0a82a-9a5d-7fb0-938f-3a04870554a4
Loaded rows: 48


### Parameterized SQL

The threshold remains data, not executable SQL. This protects the query and improves plan reuse.

In [21]:
minimum_income = 5000
query = """
SELECT Loan_ID, TotalIncome, Loan_Status
FROM loan_target
WHERE TotalIncome >= :minimum_income
ORDER BY TotalIncome DESC
LIMIT :row_limit
"""
with engine.connect() as connection:
    high_income = pd.read_sql_query(text(query), connection, params={"minimum_income": minimum_income, "row_limit": 5})
print(high_income.to_string(index=False))

 loan_id  totalincome loan_status
LP001798      10819.0           Y
LP001813      10383.0           N
LP001814       9703.0           Y
LP001811       7823.0           Y
LP001807       7550.0           Y


### Structured exit codes

A CLI entry point should translate expected failures into stable exit codes while retaining detailed logs for operators.

In [22]:
def pipeline_main(path: Path) -> int:
    try:
        frame = transform(extract(path), str(uuid7()))
        if frame.empty:
            raise PipelineError("No source rows")
        return 0
    except (ContractError, PipelineError, FileNotFoundError, ValueError):
        return 1

exit_code = pipeline_main(DATA_FILE)
assert exit_code == 0
print("Simulated CLI exit code:", exit_code)

Simulated CLI exit code: 0


## Enterprise Control

Never embed passwords in source code or logs; use environment-specific secret management.

Recommended production controls:

- Retrieve secrets from a managed secret store.
- Rotate credentials and use short-lived identity where available.
- Redact connection strings and parameters from logs.
- Grant the pipeline role only the required schemas and operations.
- Never print environment variables containing credentials.

In [23]:
audit = {
    "run_id": run_id,
    "started_at": transformed["loaded_at"].iloc[0],
    "source_file": DATA_FILE.name,
    "extracted_rows": len(raw),
    "loaded_rows": loaded_rows,
    "status": "SUCCESS",
}

assert audit["extracted_rows"] == audit["loaded_rows"]
assert "password" not in " ".join(audit).lower()
print("04 audit record:", audit)
engine.dispose()

04 audit record: {'run_id': '01a0a82a-9a5d-7fb0-938f-3a04870554a4', 'started_at': '2026-09-16T03:02:39.462465+00:00', 'source_file': 'loan_data_04.csv', 'extracted_rows': 48, 'loaded_rows': 48, 'status': 'SUCCESS'}


## PostgreSQL Execution

Run this cell after the learning and hands-on sections. It executes the same PostgreSQL pipeline used by the Python scripts, using this notebook's matched CSV partition and the active .env configuration.

The published results are available in `control.pipeline_runs`, `bronze`, `silver`, and `gold`.

In [24]:
import sys
sys.path.insert(0, str(ROOT / "src"))
from medalloan.pipeline import run as run_postgres_pipeline

# This executes the same PostgreSQL pipeline as scripts/run_pipeline.py.
# The matched CSV partition for this notebook is used as the source.
run_postgres_pipeline(DATA_FILE, load_mode="upsert")

from sqlalchemy import text
from medalloan.config import Settings
from medalloan.database import create_db_engine
validation_engine = create_db_engine(Settings.from_env())
with validation_engine.connect() as connection:
    validation = connection.execute(text("SELECT (SELECT COUNT(*) FROM bronze.loan_applications) AS bronze_rows, (SELECT COUNT(*) FROM silver.loan_applications) AS silver_rows, (SELECT COUNT(*) FROM gold.fact_loan_applications) AS gold_rows, (SELECT COUNT(*) FROM bronze.loan_applications WHERE source_file = :source_file) AS partition_rows, (SELECT COUNT(*) = COUNT(DISTINCT loan_id) FROM gold.fact_loan_applications) AS unique_ids"), {"source_file": DATA_FILE.name}).mappings().one()
print("PostgreSQL validation:", dict(validation))
print("Success criteria: partition exists, Bronze=Silver=Gold, and Gold Loan_ID unique.")
assert validation["partition_rows"] == raw["Loan_ID"].astype(str).str.strip().nunique()
assert validation["bronze_rows"] == validation["silver_rows"] == validation["gold_rows"] and validation["unique_ids"]
validation_engine.dispose()
print("PostgreSQL validation: PASSED")
print(f"PostgreSQL pipeline completed for {DATA_FILE.name}.")


PostgreSQL validation: {'bronze_rows': 381, 'silver_rows': 381, 'gold_rows': 381, 'partition_rows': 48, 'unique_ids': True}
Success criteria: partition exists, Bronze=Silver=Gold, and Gold Loan_ID unique.
PostgreSQL validation: PASSED
PostgreSQL pipeline completed for loan_data_04.csv.
